In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import tensorflow.keras.layers as tfl
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator
%matplotlib inline
np.random.seed(1)

In [ ]:
(X_train_orig, Y_train_orig), (X_test_orig, Y_test_orig) = cifar10.load_data()

# Normalize image vectors
X_train = X_train_orig/255.
X_test = X_test_orig/255.

# Reshape
Y_train = Y_train_orig
Y_test = Y_test_orig

print ("number of training examples = " + str(X_train.shape[0]))
print ("number of test examples = " + str(X_test.shape[0]))
print ("X_train shape: " + str(X_train.shape))
print ("Y_train shape: " + str(Y_train.shape))
print ("X_test shape: " + str(X_test.shape))
print ("Y_test shape: " + str(Y_test.shape))

In [ ]:
def convolutional_model(input_shape):
    input_img = tf.keras.Input(shape=input_shape)
    
    # 1ère couche conv + relu + pool
    Z1 = tfl.Conv2D(32, kernel_size=3, activation='linear', padding="same")(input_img)
    A1=tfl.BatchNormalization()(Z1)
    A1 = tfl.ReLU()(A1)
    P1 = tfl.MaxPool2D(pool_size=(2, 2), strides=(2, 2), padding='same')(A1)
   
    # 2ème couche conv + relu + pool
    Z2 = tfl.Conv2D(64, kernel_size=3, activation='linear', padding="same")(P1)
    A2=tfl.BatchNormalization()(Z2)
    A2 = tfl.ReLU()(A2)
    P2 = tfl.MaxPool2D(pool_size=(2, 2), strides=(2, 2), padding='same')(A2)

    # 3ème couche conv + relu
    Z3 = tfl.Conv2D(128, kernel_size=3, activation='linear', padding="same")(P2)
    A3=tfl.BatchNormalization()(Z3)
    A3 = tfl.ReLU()(A3)
    
    # Flatten + dense
    F = tfl.Flatten()(A3)
    D1 = tfl.Dense(128, activation='relu')(F) # couche dense intermédiaire pour plus de param
    D1 = tfl.Dropout(0.3)(D1)
    outputs = tfl.Dense(10, activation='softmax')(D1)

    model = tf.keras.Model(inputs=input_img, outputs=outputs)
    return model

In [ ]:
conv_model = convolutional_model(input_shape=(32, 32, 3))
conv_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
conv_model.summary()

In [ ]:
Y_train = np.squeeze(Y_train)  # retire les dimensions de taille 1
Y_test = np.squeeze(Y_test)


train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train)).batch(64)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, Y_test)).batch(64)
history = conv_model.fit(train_dataset, epochs=30, validation_data=test_dataset)

In [ ]:
# Courbe Loss
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.legend()
plt.title("Loss")
plt.show()

# Courbe Accuracy
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.legend()
plt.title("Accuracy")
plt.show()

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("modele.keras")
print("Modèle rechargé !")

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.preprocessing import image

# Chemin vers l’image
image_path = r"C:\Users\wassim\Downloads\b_2.jpg"

# Charger l'image et redimensionner à 32x32
img = image.load_img(image_path, target_size=(32, 32))

# Convertir en tableau numpy
img_array = image.img_to_array(img)

# Normaliser
img_array = img_array / 255.0

# Ajouter une dimension batch (1 image)
img_array = np.expand_dims(img_array, axis=0)

In [ ]:
# Prédiction avec le modèle
pred = conv_model.predict(img_array)
pred_class = np.argmax(pred, axis=1)

# Afficher le résultat
cifar_classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print("Prédiction :", cifar_classes[pred_class[0]])